# Import Libraries

In [ ]:
import pandas as pd # for data manipulation and analysis
import numpy as np  # for numerical computation


# Load Datasets

In [ ]:
matches = pd.read_csv("/content/matches.csv")
deliveries = pd.read_csv("/content/deliveries.csv")

print("Dataset loaded")

Dataset loaded


## Dataset Overview (Matches)

In [ ]:
print(matches.head(3))
print(matches.shape)

   id  season       city        date                team1  \
0   1    2017  Hyderabad  2017-04-05  Sunrisers Hyderabad   
1   2    2017       Pune  2017-04-06       Mumbai Indians   
2   3    2017     Rajkot  2017-04-07        Gujarat Lions   

                         team2                  toss_winner toss_decision  \
0  Royal Challengers Bangalore  Royal Challengers Bangalore         field   
1       Rising Pune Supergiant       Rising Pune Supergiant         field   
2        Kolkata Knight Riders        Kolkata Knight Riders         field   

   result  dl_applied                  winner  win_by_runs  win_by_wickets  \
0  normal           0     Sunrisers Hyderabad           35               0   
1  normal           0  Rising Pune Supergiant            0               7   
2  normal           0   Kolkata Knight Riders            0              10   

  player_of_match                                      venue         umpire1  \
0    Yuvraj Singh  Rajiv Gandhi International Stadium

In [ ]:
print(matches.columns.tolist())

['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']


In [ ]:
print(matches.isnull().sum())

id                   0
season               0
city                 7
date                 0
team1                0
team2                0
toss_winner          0
toss_decision        0
result               0
dl_applied           0
winner               3
win_by_runs          0
win_by_wickets       0
player_of_match      3
venue                0
umpire1              1
umpire2              1
umpire3            636
dtype: int64


## Dataset Overview (Deliveries)

In [ ]:
print(deliveries.head(3))
print(deliveries.shape)

   match_id  inning         batting_team                 bowling_team  over  \
0         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
1         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
2         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   

   ball    batsman non_striker    bowler  is_super_over  ...  bye_runs  \
0     1  DA Warner    S Dhawan  TS Mills              0  ...         0   
1     2  DA Warner    S Dhawan  TS Mills              0  ...         0   
2     3  DA Warner    S Dhawan  TS Mills              0  ...         0   

   legbye_runs  noball_runs  penalty_runs  batsman_runs  extra_runs  \
0            0            0             0             0           0   
1            0            0             0             0           0   
2            0            0             0             4           0   

   total_runs  player_dismissed dismissal_kind fielder  
0           0               NaN            N

In [ ]:
print(deliveries.columns.tolist())

['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']


In [ ]:
print(deliveries.isnull().sum())

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batsman                  0
non_striker              0
bowler                   0
is_super_over            0
wide_runs                0
bye_runs                 0
legbye_runs              0
noball_runs              0
penalty_runs             0
batsman_runs             0
extra_runs               0
total_runs               0
player_dismissed    143022
dismissal_kind      143022
fielder             145091
dtype: int64


# Clean matches

In [ ]:
matches_clean = matches.drop(columns=['umpire3', 'player_of_match', 'umpire1', 'umpire2']) # drop unnecessary column

In [ ]:
matches_clean = matches_clean.dropna(subset=['winner'])  # drop no-result matches

In [ ]:
matches_clean['city'] = matches_clean['city'].fillna(matches_clean['city'].mode()[0])

In [ ]:
print("Matches after cleaning:", matches_clean.shape)
print("Null check:\n", matches_clean.isnull().sum())

Matches after cleaning: (633, 14)
Null check:
 id                0
season            0
city              0
date              0
team1             0
team2             0
toss_winner       0
toss_decision     0
result            0
dl_applied        0
winner            0
win_by_runs       0
win_by_wickets    0
venue             0
dtype: int64


# Clean deliveries

In [ ]:
deliveries_clean = deliveries[deliveries['is_super_over'] == 0]

In [ ]:
print("Deliveries after removing super overs:", deliveries_clean.shape)

Deliveries after removing super overs: (150379, 21)


In [ ]:
print("Null check:\n", deliveries_clean.isnull().sum())

Null check:
 match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batsman                  0
non_striker              0
bowler                   0
is_super_over            0
wide_runs                0
bye_runs                 0
legbye_runs              0
noball_runs              0
penalty_runs             0
batsman_runs             0
extra_runs               0
total_runs               0
player_dismissed    142955
dismissal_kind      142955
fielder             145018
dtype: int64


### Feature Engineering: Converting Ball-Level Data into Over-Level Statistics

In [ ]:
over_data = deliveries_clean.groupby(['match_id', 'inning', 'over']).agg(
    runs_in_over      = ('total_runs', 'sum'),
    wickets_in_over   = ('player_dismissed', lambda x: x.notna().sum()),
    extras_in_over    = ('extra_runs', 'sum'),
    fours_in_over     = ('batsman_runs', lambda x: (x == 4).sum()),
    sixes_in_over     = ('batsman_runs', lambda x: (x == 6).sum()),
    dots_in_over      = ('total_runs', lambda x: (x == 0).sum()),
    batting_team      = ('batting_team', 'first'),
    bowling_team      = ('bowling_team', 'first')
).reset_index()


print("Over level shape:", over_data.shape)

Over level shape: (24374, 11)


In [ ]:
over_data.head(10)

,match_id,inning,over,runs_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,dots_in_over,batting_team,bowling_team
0,1,1,1,7,0,3,1,0,4,Sunrisers Hyderabad,Royal Challengers Bangalore
1,1,1,2,16,1,1,2,1,2,Sunrisers Hyderabad,Royal Challengers Bangalore
2,1,1,3,6,0,0,0,0,2,Sunrisers Hyderabad,Royal Challengers Bangalore
3,1,1,4,4,0,0,0,0,2,Sunrisers Hyderabad,Royal Challengers Bangalore
4,1,1,5,9,0,0,1,0,1,Sunrisers Hyderabad,Royal Challengers Bangalore
5,1,1,6,17,0,0,4,0,1,Sunrisers Hyderabad,Royal Challengers Bangalore
6,1,1,7,5,0,0,0,0,1,Sunrisers Hyderabad,Royal Challengers Bangalore
7,1,1,8,11,0,0,0,1,0,Sunrisers Hyderabad,Royal Challengers Bangalore
8,1,1,9,9,0,0,0,0,0,Sunrisers Hyderabad,Royal Challengers Bangalore
9,1,1,10,4,0,0,0,0,2,Sunrisers Hyderabad,Royal Challengers Bangalore


### Feature Engineering: Creating Derived Match Features for Over-Level Analysis

In [ ]:
over_data = over_data.sort_values(['match_id', 'inning', 'over']).reset_index(drop=True)

over_data['total_runs']    = over_data.groupby(['match_id', 'inning'])['runs_in_over'].cumsum()
over_data['total_wickets'] = over_data.groupby(['match_id', 'inning'])['wickets_in_over'].cumsum()
over_data['total_extras']  = over_data.groupby(['match_id', 'inning'])['extras_in_over'].cumsum()
over_data['total_fours']   = over_data.groupby(['match_id', 'inning'])['fours_in_over'].cumsum()
over_data['total_sixes']   = over_data.groupby(['match_id', 'inning'])['sixes_in_over'].cumsum()
over_data['total_dots']    = over_data.groupby(['match_id', 'inning'])['dots_in_over'].cumsum()


In [ ]:
print(over_data.shape)

(24374, 17)


In [ ]:

over_data.head(10)

,match_id,inning,over,runs_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,dots_in_over,batting_team,bowling_team,total_runs,total_wickets,total_extras,total_fours,total_sixes,total_dots
0,1,1,1,7,0,3,1,0,4,Sunrisers Hyderabad,Royal Challengers Bangalore,7,0,3,1,0,4
1,1,1,2,16,1,1,2,1,2,Sunrisers Hyderabad,Royal Challengers Bangalore,23,1,4,3,1,6
2,1,1,3,6,0,0,0,0,2,Sunrisers Hyderabad,Royal Challengers Bangalore,29,1,4,3,1,8
3,1,1,4,4,0,0,0,0,2,Sunrisers Hyderabad,Royal Challengers Bangalore,33,1,4,3,1,10
4,1,1,5,9,0,0,1,0,1,Sunrisers Hyderabad,Royal Challengers Bangalore,42,1,4,4,1,11
5,1,1,6,17,0,0,4,0,1,Sunrisers Hyderabad,Royal Challengers Bangalore,59,1,4,8,1,12
6,1,1,7,5,0,0,0,0,1,Sunrisers Hyderabad,Royal Challengers Bangalore,64,1,4,8,1,13
7,1,1,8,11,0,0,0,1,0,Sunrisers Hyderabad,Royal Challengers Bangalore,75,1,4,8,2,13
8,1,1,9,9,0,0,0,0,0,Sunrisers Hyderabad,Royal Challengers Bangalore,84,1,4,8,2,13
9,1,1,10,4,0,0,0,0,2,Sunrisers Hyderabad,Royal Challengers Bangalore,88,1,4,8,2,15


### Merging Over-Level Data with Match-Level Information

In [ ]:
# Select only the important match-level columns
match_cols = [
    'id', 'team1', 'team2', 'toss_winner', 'toss_decision',
    'winner', 'venue', 'city', 'season', 'dl_applied'
]

# Merge over-level data with match-level information
df = over_data.merge(
    matches_clean[match_cols],
    left_on='match_id',
    right_on='id',
    how='left'
)

# Remove duplicate column after merge
df.drop(columns='id', inplace=True)

# Check the result
print("Final dataset shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())


Final dataset shape: (24374, 26)

Missing values per column:
 match_id            0
inning              0
over                0
runs_in_over        0
wickets_in_over     0
extras_in_over      0
fours_in_over       0
sixes_in_over       0
dots_in_over        0
batting_team        0
bowling_team        0
total_runs          0
total_wickets       0
total_extras        0
total_fours         0
total_sixes         0
total_dots          0
team1              53
team2              53
toss_winner        53
toss_decision      53
winner             53
venue              53
city               53
season             53
dl_applied         53
dtype: int64


In [ ]:
print("\nSample data:\n", df.head(3))


Sample data:
    match_id  inning  over  runs_in_over  wickets_in_over  extras_in_over  \
0         1       1     1             7                0               3   
1         1       1     2            16                1               1   
2         1       1     3             6                0               0   

   fours_in_over  sixes_in_over  dots_in_over         batting_team  ...  \
0              1              0             4  Sunrisers Hyderabad  ...   
1              2              1             2  Sunrisers Hyderabad  ...   
2              0              0             2  Sunrisers Hyderabad  ...   

  total_dots                team1                        team2  \
0          4  Sunrisers Hyderabad  Royal Challengers Bangalore   
1          6  Sunrisers Hyderabad  Royal Challengers Bangalore   
2          8  Sunrisers Hyderabad  Royal Challengers Bangalore   

                   toss_winner  toss_decision               winner  \
0  Royal Challengers Bangalore          fiel

# Cleaning merged data

In [ ]:
# Drop rows where match info didn't merge (abandoned matches)
df = df.dropna(subset=['winner', 'team1', 'team2'])
print("After dropping unmatched:", df.shape)

After dropping unmatched: (24321, 26)


In [ ]:
print(df.isnull().sum())  # Null check

match_id           0
inning             0
over               0
runs_in_over       0
wickets_in_over    0
extras_in_over     0
fours_in_over      0
sixes_in_over      0
dots_in_over       0
batting_team       0
bowling_team       0
total_runs         0
total_wickets      0
total_extras       0
total_fours        0
total_sixes        0
total_dots         0
team1              0
team2              0
toss_winner        0
toss_decision      0
winner             0
venue              0
city               0
season             0
dl_applied         0
dtype: int64
